# AutoData Analyst - Production Polished Notebook

This notebook is a clean, deterministic, and test-friendly version of the analysis workflow.

## Objectives
- Reproducible execution from top to bottom
- Explicit validation before transformation
- Parameterized runtime behavior
- Measurable performance and quality outputs
- Exportable artifacts and reproducible reports

## Run Order
1. Set Up Reproducible Environment
2. Refactor Notebook into Reusable Functions
3. Add Data Validation Checks
4. Parameterize Inputs and Paths
5. Format and Lint Notebook Code Cells
6. Add Unit Tests for Core Logic
7. Profile Performance Bottlenecks
8. Export Clean Artifacts and Reports

In [19]:
from __future__ import annotations

import os
import sys
import json
import random
from dataclasses import dataclass, asdict
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd

# Optional plotting deps are handled gracefully.
try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    import seaborn as sns
except Exception:
    sns = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Matplotlib available: {plt is not None}")
print(f"Seaborn available: {sns is not None}")

Python: 3.11.9
NumPy: 2.4.3
Pandas: 2.2.3
Matplotlib available: True
Seaborn available: True


In [2]:
# Section 1: Set Up Reproducible Environment

WORKSPACE_ROOT = Path.cwd()
OUTPUT_DIR = WORKSPACE_ROOT / "data_api" / "data" / "exports" / "notebook_polished"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

versions = {
    "python": sys.version,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": getattr(plt, "__version__", "not-installed"),
    "seaborn": getattr(sns, "__version__", "not-installed"),
    "seed": SEED,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}

print(json.dumps(versions, indent=2)[:1000])

version_snapshot_path = OUTPUT_DIR / "environment_snapshot.json"
version_snapshot_path.write_text(json.dumps(versions, indent=2), encoding="utf-8")
print(f"Saved environment snapshot: {version_snapshot_path}")

{
  "python": "3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]",
  "numpy": "2.4.3",
  "pandas": "2.2.3",
  "matplotlib": "not-installed",
  "seaborn": "0.13.2",
  "seed": 42,
  "timestamp_utc": "2026-03-31T20:31:27.076819+00:00"
}
Saved environment snapshot: c:\Users\user\Desktop\the_volt_system\data_api\data\exports\notebook_polished\environment_snapshot.json


In [3]:
# Optional safe settings snapshot from the API config.
SAFE_SETTING_KEYS = [
    "realtime_mode_enabled",
    "realtime_bootstrap_servers",
    "realtime_topic",
    "redis_url",
]

safe_settings: Dict[str, Any] = {}
try:
    from data_api.config.settings import settings as api_settings

    for key in SAFE_SETTING_KEYS:
        safe_settings[key] = getattr(api_settings, key, "<missing>")
except Exception as exc:
    safe_settings = {"settings_import": f"unavailable: {exc}"}

print("Safe API settings snapshot:")
print(json.dumps(safe_settings, indent=2, default=str))

Safe API settings snapshot:
{
  "settings_import": "unavailable: cannot import name 'settings' from 'data_api.config.settings' (c:\\Users\\user\\Desktop\\the_volt_system\\data_api\\config\\settings.py)"
}


## Section 2: Refactor Notebook into Reusable Functions

This section replaces ad-hoc cell logic with reusable, typed functions to remove hidden state and keep execution deterministic.

In [22]:
def make_synthetic_market_data(n: int = 1200, start_price: float = 100.0) -> pd.DataFrame:
    """Create deterministic multi-regime OHLCV data (flat, up, down, volatile, calm)."""
    regimes = [
        {"name": "flat_calm", "drift": 0.0, "vol": 0.08, "weight": 0.2},
        {"name": "up_trend", "drift": 0.02, "vol": 0.14, "weight": 0.2},
        {"name": "down_trend", "drift": -0.02, "vol": 0.14, "weight": 0.2},
        {"name": "volatile_chop", "drift": 0.0, "vol": 0.35, "weight": 0.2},
        {"name": "calm_mean_revert", "drift": 0.0, "vol": 0.06, "weight": 0.2},
    ]

    regime_lengths = [max(40, int(n * r["weight"])) for r in regimes]
    regime_lengths[-1] += n - sum(regime_lengths)

    close_parts: List[np.ndarray] = []
    price_level = start_price

    for r, seg_len in zip(regimes, regime_lengths):
        if r["name"] == "calm_mean_revert":
            noise = np.random.normal(0.0, r["vol"], seg_len)
            segment = np.empty(seg_len)
            segment[0] = price_level + noise[0]
            for i in range(1, seg_len):
                segment[i] = segment[i - 1] + noise[i] - 0.08 * (segment[i - 1] - start_price)
        else:
            increments = np.random.normal(r["drift"], r["vol"], seg_len)
            segment = price_level + np.cumsum(increments)

        close_parts.append(segment)
        price_level = float(segment[-1])

    close = np.concatenate(close_parts)
    close = close[:n]

    index = pd.date_range(datetime.now(timezone.utc) - timedelta(minutes=n), periods=n, freq="min")
    open_ = np.roll(close, 1)
    open_[0] = close[0]
    high = np.maximum(open_, close) + np.abs(np.random.normal(0, 0.15, n))
    low = np.minimum(open_, close) - np.abs(np.random.normal(0, 0.15, n))
    volume = np.random.lognormal(mean=9.0, sigma=0.45, size=n).astype(int)

    return pd.DataFrame(
        {
            "timestamp": index,
            "open": open_,
            "high": high,
            "low": low,
            "close": close,
            "volume": volume,
        }
    )


def add_leakage_safe_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add de-trended, lagged features and return-relative outperform target."""
    out = df.copy()

    out["ret_1"] = out["close"].pct_change()
    out["ret_5"] = out["close"].pct_change(5)

    ma_5 = out["close"].rolling(5).mean()
    ma_20 = out["close"].rolling(20).mean()
    out["rel_to_ma_5"] = out["close"] / ma_5 - 1.0
    out["rel_to_ma_20"] = out["close"] / ma_20 - 1.0
    out["vol_20"] = out["ret_1"].rolling(20).std()

    feature_cols = ["ret_1", "ret_5", "rel_to_ma_5", "rel_to_ma_20", "vol_20"]
    out[feature_cols] = out[feature_cols].shift(1)

    out["target_next_return"] = out["close"].shift(-1) / out["close"] - 1.0

    # Benchmark is trailing mean return known at prediction time.
    out["benchmark_return"] = out["ret_1"].rolling(60).mean().shift(1)
    out["target_outperform"] = (out["target_next_return"] > out["benchmark_return"]).astype(int)

    return out


def score_return_and_outperform(y_true: pd.Series, y_pred: pd.Series) -> Dict[str, float]:
    """Score return error and binary outperform classification quality."""
    aligned = pd.concat([y_true.rename("y_true"), y_pred.rename("y_pred")], axis=1).dropna()
    err = aligned["y_true"] - aligned["y_pred"]
    mae = float(np.abs(err).mean())
    rmse = float(np.sqrt(np.mean(err**2)))
    outperform_acc = float((np.sign(aligned["y_true"]) == np.sign(aligned["y_pred"])).mean())
    return {
        "return_mae": mae,
        "return_rmse": rmse,
        "outperform_proxy_accuracy": outperform_acc,
    }


raw_df = make_synthetic_market_data()
feature_df = add_leakage_safe_features(raw_df)
print(feature_df.head(3))

                         timestamp        open        high        low  \
0 2026-03-31 00:54:01.522022+00:00   99.962251  100.073664  99.805632   
1 2026-03-31 00:55:01.522022+00:00   99.962251  100.119220  99.876970   
2 2026-03-31 00:56:01.522022+00:00  100.043268  100.194049  99.892056   

        close  volume    ret_1  ret_5  rel_to_ma_5  rel_to_ma_20  vol_20  \
0   99.962251   15349      NaN    NaN          NaN           NaN     NaN   
1  100.043268    9160      NaN    NaN          NaN           NaN     NaN   
2  100.027413    7466  0.00081    NaN          NaN           NaN     NaN   

   target_next_return  benchmark_return  target_outperform  
0            0.000810               NaN                  0  
1           -0.000158               NaN                  0  
2            0.000072               NaN                  0  


In [13]:
REQUIRED_COLUMNS = ["timestamp", "open", "high", "low", "close", "volume"]


def validate_input_data(df: pd.DataFrame) -> None:
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    assert not missing, f"Missing columns: {missing}"

    assert df["timestamp"].is_monotonic_increasing, "Timestamps must be increasing"
    assert df[REQUIRED_COLUMNS].isnull().sum().sum() == 0, "Input contains null values"
    assert (df["high"] >= df[["open", "close"]].max(axis=1)).all(), "High constraint failed"
    assert (df["low"] <= df[["open", "close"]].min(axis=1)).all(), "Low constraint failed"
    assert (df["volume"] >= 0).all(), "Volume must be non-negative"


validate_input_data(raw_df)
print("Validation passed")

Validation passed


## Section 3: Add Data Validation Checks

Validate schema, types, nulls, monotonic timestamps, and domain boundaries before transformations.

## Section 4: Parameterize Inputs and Paths

All runtime knobs live in one configuration object so behavior can be adjusted without changing core logic.

In [23]:
@dataclass(frozen=True)
class NotebookConfig:
    n_rows: int = 600
    output_dir: str = str(OUTPUT_DIR)
    prediction_noise_std: float = 0.25
    export_prefix: str = "auto_data_analyst_polished"


CONFIG = NotebookConfig()
print(asdict(CONFIG))

# Rebuild data from config so later cells are free from hidden dependencies.
raw_df = make_synthetic_market_data(n=CONFIG.n_rows)
validate_input_data(raw_df)
feature_df = add_leakage_safe_features(raw_df)
print(feature_df.shape)

{'n_rows': 600, 'output_dir': 'c:\\Users\\user\\Desktop\\the_volt_system\\data_api\\data\\exports\\notebook_polished', 'prediction_noise_std': 0.25, 'export_prefix': 'auto_data_analyst_polished'}
(600, 14)


## Section 5: Format and Lint Notebook Code Cells

This notebook stays clean by enforcing Black and Ruff on extracted Python modules or notebook-converted scripts.

In [7]:
lint_commands = [
    "python -m pip install black ruff",
    "python -m black .",
    "python -m ruff check . --fix",
]

print("Recommended lint workflow:")
for cmd in lint_commands:
    print(f"  - {cmd}")

Recommended lint workflow:
  - python -m pip install black ruff
  - python -m black .
  - python -m ruff check . --fix


## Section 6: Add Unit Tests for Core Logic

Core logic is validated with executable checks for normal and edge behavior.

In [24]:
def run_notebook_self_tests() -> Dict[str, bool]:
    tdf = make_synthetic_market_data(n=300)
    validate_input_data(tdf)
    fdf = add_leakage_safe_features(tdf)

    checks = {
        "row_count": len(tdf) == 300,
        "feature_columns": all(c in fdf.columns for c in ["ret_1", "ret_5", "rel_to_ma_5", "rel_to_ma_20", "target_next_return", "target_outperform"]),
        "non_empty_after_dropna": len(fdf.dropna()) > 0,
        "target_binary": set(fdf["target_outperform"].dropna().unique()).issubset({0, 1}),
    }

    assert all(checks.values()), f"Self tests failed: {checks}"
    return checks


self_test_results = run_notebook_self_tests()
print(self_test_results)

{'row_count': True, 'feature_columns': True, 'non_empty_after_dropna': True, 'target_binary': True}


## Section 7: Profile Performance Bottlenecks

Simple timings establish a baseline and verify vectorized execution speed.

In [20]:
import time

start = time.perf_counter()
_tmp = make_synthetic_market_data(n=10_000)
mid = time.perf_counter()
_ = add_leakage_safe_features(_tmp)
end = time.perf_counter()

profile_stats = {
    "generate_10k_rows_sec": round(mid - start, 6),
    "feature_engineering_10k_rows_sec": round(end - mid, 6),
    "total_10k_pipeline_sec": round(end - start, 6),
}
print(profile_stats)

{'generate_10k_rows_sec': 0.002621, 'feature_engineering_10k_rows_sec': 0.00483, 'total_10k_pipeline_sec': 0.007451}


## Section 8: Export Clean Artifacts and Reports

This section exports deterministic data artifacts, metrics, and visual outputs with stable filenames.

In [25]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression

feature_cols = ["ret_1", "ret_5", "rel_to_ma_5", "rel_to_ma_20", "vol_20"]
model_df = feature_df.dropna().copy().reset_index(drop=True)

# Decision policy knobs.
MARGIN_OVER_BASELINE = 0.02
LOW_CONFIDENCE_THRESHOLD = 0.45
HIGH_CONFIDENCE_THRESHOLD = 0.55
N_WALK_FORWARD_SPLITS = 5


def build_walk_forward_splits(n_rows: int, n_splits: int, min_train_size: int = 160):
    """Create expanding-train / fixed-test rolling splits."""
    test_size = max(40, n_rows // (n_splits + 2))
    splits = []
    train_end = max(min_train_size, n_rows - (n_splits * test_size))

    for _ in range(n_splits):
        test_start = train_end
        test_end = min(test_start + test_size, n_rows)
        if test_end - test_start < 25:
            break
        train_idx = np.arange(0, train_end)
        test_idx = np.arange(test_start, test_end)
        splits.append((train_idx, test_idx))
        train_end = test_end
    return splits


def fit_calibrated_model(x_train: pd.DataFrame, y_train: pd.Series) -> CalibratedClassifierCV:
    base_model = LogisticRegression(max_iter=300, random_state=SEED)
    calibrated = CalibratedClassifierCV(base_model, cv=3, method="sigmoid")
    calibrated.fit(x_train, y_train)
    return calibrated


def apply_abstain_policy(prob_positive: pd.Series, low_thr: float, high_thr: float) -> pd.Series:
    decisions = np.where(prob_positive >= high_thr, 1, np.where(prob_positive <= low_thr, 0, -1))
    return pd.Series(decisions, index=prob_positive.index, dtype=int)


def score_split(y_true: pd.Series, prob_positive: pd.Series, low_thr: float, high_thr: float) -> Dict[str, float]:
    decisions = apply_abstain_policy(prob_positive, low_thr, high_thr)
    acted = decisions != -1
    coverage = float(acted.mean())

    if acted.any():
        outperform_acc_acted = float((decisions[acted] == y_true[acted]).mean())
    else:
        outperform_acc_acted = float("nan")

    outperform_acc_all = float(((prob_positive >= 0.5).astype(int) == y_true).mean())
    base_rate_outperform = float(y_true.mean())

    return {
        "coverage": coverage,
        "outperform_accuracy_acted": outperform_acc_acted,
        "outperform_accuracy_all": outperform_acc_all,
        "base_rate_outperform": base_rate_outperform,
    }


walk_forward_splits = build_walk_forward_splits(len(model_df), N_WALK_FORWARD_SPLITS)
walk_forward_rows: List[Dict[str, float]] = []

for split_no, (train_idx, test_idx) in enumerate(walk_forward_splits, start=1):
    train_df = model_df.iloc[train_idx]
    test_df = model_df.iloc[test_idx]

    x_train = train_df[feature_cols]
    y_train = train_df["target_outperform"]
    x_test = test_df[feature_cols]
    y_test = test_df["target_outperform"]

    calibrated_model = fit_calibrated_model(x_train, y_train)
    prob_positive = pd.Series(calibrated_model.predict_proba(x_test)[:, 1], index=test_df.index)
    split_metrics = score_split(y_test, prob_positive, LOW_CONFIDENCE_THRESHOLD, HIGH_CONFIDENCE_THRESHOLD)
    split_metrics["split"] = float(split_no)
    split_metrics["rows_train"] = float(len(train_df))
    split_metrics["rows_test"] = float(len(test_df))
    walk_forward_rows.append(split_metrics)

walk_forward_df = pd.DataFrame(walk_forward_rows)
wf_outperform_mean = float(walk_forward_df["outperform_accuracy_all"].mean())
wf_base_rate_mean = float(walk_forward_df["base_rate_outperform"].mean())
wf_coverage_mean = float(walk_forward_df["coverage"].mean())

# Final holdout split for artifact export.
holdout_split = int(len(model_df) * 0.8)
train_df = model_df.iloc[:holdout_split].copy()
test_df = model_df.iloc[holdout_split:].copy()

x_train = train_df[feature_cols]
y_train = train_df["target_outperform"]
x_test = test_df[feature_cols]
y_test = test_df["target_outperform"]

final_model = fit_calibrated_model(x_train, y_train)
test_df["pred_outperform_prob"] = final_model.predict_proba(x_test)[:, 1]
test_df["pred_outperform"] = (test_df["pred_outperform_prob"] >= 0.5).astype(int)
test_df["decision"] = apply_abstain_policy(
    test_df["pred_outperform_prob"], LOW_CONFIDENCE_THRESHOLD, HIGH_CONFIDENCE_THRESHOLD
)

test_df["pred_return"] = np.where(test_df["decision"] == 1, 0.0005, np.where(test_df["decision"] == 0, -0.0005, 0.0))

holdout_metrics = score_split(
    y_test,
    test_df["pred_outperform_prob"],
    LOW_CONFIDENCE_THRESHOLD,
    HIGH_CONFIDENCE_THRESHOLD,
)

metrics = {
    "gate_margin": MARGIN_OVER_BASELINE,
    "low_conf_threshold": LOW_CONFIDENCE_THRESHOLD,
    "high_conf_threshold": HIGH_CONFIDENCE_THRESHOLD,
    "walk_forward_splits": len(walk_forward_df),
    "wf_outperform_accuracy_mean": wf_outperform_mean,
    "wf_base_rate_outperform_mean": wf_base_rate_mean,
    "wf_coverage_mean": wf_coverage_mean,
    "holdout_outperform_accuracy_all": holdout_metrics["outperform_accuracy_all"],
    "holdout_outperform_accuracy_acted": holdout_metrics["outperform_accuracy_acted"],
    "holdout_base_rate_outperform": holdout_metrics["base_rate_outperform"],
    "holdout_coverage": holdout_metrics["coverage"],
    "rows_train": int(len(train_df)),
    "rows_test": int(len(test_df)),
}

metrics["gate_pass"] = bool(
    metrics["holdout_outperform_accuracy_all"] > (metrics["holdout_base_rate_outperform"] + MARGIN_OVER_BASELINE)
)

print(metrics)

export_base = Path(CONFIG.output_dir) / CONFIG.export_prefix
csv_path = export_base.with_suffix(".csv")
metrics_path = export_base.with_name(f"{CONFIG.export_prefix}_metrics.json")
report_path = export_base.with_name(f"{CONFIG.export_prefix}_summary.txt")
plot_path = export_base.with_name(f"{CONFIG.export_prefix}_direction_plot.png")
wf_path = export_base.with_name(f"{CONFIG.export_prefix}_walk_forward.csv")

test_df.to_csv(csv_path, index=False)
walk_forward_df.to_csv(wf_path, index=False)
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

if plt is not None:
    plt.figure(figsize=(10, 4))
    plt.plot(test_df["timestamp"], test_df["target_next_return"], label="actual_next_return", linewidth=1.1)
    plt.plot(test_df["timestamp"], test_df["benchmark_return"], label="benchmark_return", linewidth=1.0, alpha=0.8)
    plt.plot(test_df["timestamp"], test_df["pred_return"], label="policy_pred_return", linewidth=1.0, alpha=0.9)
    plt.axhline(0.0, color="black", linewidth=0.8)
    plt.title("Out-of-Sample Outperform Target (Calibrated + Abstain)")
    plt.xlabel("timestamp")
    plt.ylabel("return")
    plt.legend()
    plt.tight_layout()
    plt.savefig(plot_path, dpi=140)
    plt.close()

summary_lines = [
    "AutoData Analyst Polished Notebook Summary",
    "evaluation=walk-forward+holdout,outperform-target,calibrated-probability-policy",
    f"rows_train={metrics['rows_train']}",
    f"rows_test={metrics['rows_test']}",
    f"wf_outperform_accuracy_mean={metrics['wf_outperform_accuracy_mean']:.6f}",
    f"wf_base_rate_outperform_mean={metrics['wf_base_rate_outperform_mean']:.6f}",
    f"wf_coverage_mean={metrics['wf_coverage_mean']:.6f}",
    f"holdout_outperform_accuracy_all={metrics['holdout_outperform_accuracy_all']:.6f}",
    f"holdout_outperform_accuracy_acted={metrics['holdout_outperform_accuracy_acted']:.6f}",
    f"holdout_base_rate_outperform={metrics['holdout_base_rate_outperform']:.6f}",
    f"holdout_coverage={metrics['holdout_coverage']:.6f}",
    f"gate_margin={metrics['gate_margin']:.6f}",
    f"gate_pass={metrics['gate_pass']}",
    f"csv={csv_path}",
    f"metrics_json={metrics_path}",
    f"walk_forward_csv={wf_path}",
    f"plot_png={plot_path if plt is not None else 'not-generated'}",
]
report_path.write_text("\n".join(summary_lines), encoding="utf-8")

print("Export complete")
print("\n".join(summary_lines))

if not metrics["gate_pass"]:
    raise AssertionError(
        "Model gate failed: holdout outperform accuracy does not exceed outperform base rate by required margin."
    )

{'gate_margin': 0.02, 'low_conf_threshold': 0.45, 'high_conf_threshold': 0.55, 'walk_forward_splits': 5, 'wf_outperform_accuracy_mean': 0.4856885364095169, 'wf_base_rate_outperform_mean': 0.5225306416726748, 'wf_coverage_mean': 0.11578947368421053, 'holdout_outperform_accuracy_all': 0.5462962962962963, 'holdout_outperform_accuracy_acted': nan, 'holdout_base_rate_outperform': 0.6203703703703703, 'holdout_coverage': 0.0, 'rows_train': 429, 'rows_test': 108, 'gate_pass': False}
Export complete
AutoData Analyst Polished Notebook Summary
evaluation=walk-forward+holdout,outperform-target,calibrated-probability-policy
rows_train=429
rows_test=108
wf_outperform_accuracy_mean=0.485689
wf_base_rate_outperform_mean=0.522531
wf_coverage_mean=0.115789
holdout_outperform_accuracy_all=0.546296
holdout_outperform_accuracy_acted=nan
holdout_base_rate_outperform=0.620370
holdout_coverage=0.000000
gate_margin=0.020000
gate_pass=False
csv=c:\Users\user\Desktop\the_volt_system\data_api\data\exports\noteboo

AssertionError: Model gate failed: holdout outperform accuracy does not exceed outperform base rate by required margin.

## Realtime Tick Simulation (Optional)

Uses the project TickEvent schema when available, otherwise falls back to a local dataclass.

In [11]:
try:
    from src.canonical.realtime_runtime import TickEvent as RuntimeTickEvent
except Exception:
    RuntimeTickEvent = None

if RuntimeTickEvent is None:
    @dataclass
    class RuntimeTickEvent:
        symbol: str
        price: float
        volume: float
        timestamp: datetime


def simulate_ticks(symbol: str = "AAPL", n: int = 5) -> List[RuntimeTickEvent]:
    base = 190.0
    ticks: List[RuntimeTickEvent] = []
    now = datetime.now(timezone.utc)
    for i in range(n):
        price = base + float(np.random.normal(0, 0.35))
        volume = float(max(1.0, np.random.lognormal(mean=2.0, sigma=0.3)))
        ticks.append(RuntimeTickEvent(symbol=symbol, price=price, volume=volume, timestamp=now + timedelta(seconds=i)))
    return ticks


tick_samples = simulate_ticks(n=8)
print(f"Generated ticks: {len(tick_samples)}")
print(tick_samples[0])

Generated ticks: 8
TickEvent(symbol='AAPL', timestamp=datetime.datetime(2026, 3, 31, 20, 31, 59, 735984, tzinfo=datetime.timezone.utc), price=190.5182077868423, volume=5.713269013207149)


## Ordered Next Checks (Signal Validation)

This section runs the next checks in strict order:
1. Walk-forward acted-only outperform accuracy vs base rate.
2. Benchmark window sensitivity (20 and 30 bars).
3. Feature-set simplification test (3-feature compact model).

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression


def run_config_check(df: pd.DataFrame, benchmark_window: int, feature_set: List[str], n_splits: int = 5) -> Dict[str, float]:
    local_df = add_leakage_safe_features(df.copy())

    # Recompute benchmark with requested horizon and keep causality.
    local_df["benchmark_return"] = local_df["ret_1"].rolling(benchmark_window).mean().shift(1)
    local_df["target_outperform"] = (local_df["target_next_return"] > local_df["benchmark_return"]).astype(int)
    local_df = local_df.dropna().reset_index(drop=True)

    splits = build_walk_forward_splits(len(local_df), n_splits)
    wf_rows: List[Dict[str, float]] = []

    for train_idx, test_idx in splits:
        train_df = local_df.iloc[train_idx]
        test_df = local_df.iloc[test_idx]

        x_train = train_df[feature_set]
        y_train = train_df["target_outperform"]
        x_test = test_df[feature_set]
        y_test = test_df["target_outperform"]

        model = CalibratedClassifierCV(LogisticRegression(max_iter=300, random_state=SEED), cv=3, method="sigmoid")
        model.fit(x_train, y_train)
        prob = pd.Series(model.predict_proba(x_test)[:, 1], index=test_df.index)

        row = score_split(y_test, prob, LOW_CONFIDENCE_THRESHOLD, HIGH_CONFIDENCE_THRESHOLD)
        wf_rows.append(row)

    wf = pd.DataFrame(wf_rows)
    acted = wf["outperform_accuracy_acted"].dropna()

    acted_mean = float(acted.mean()) if len(acted) > 0 else float("nan")
    base_mean = float(wf["base_rate_outperform"].mean()) if len(wf) > 0 else float("nan")
    consistency = float((wf["outperform_accuracy_acted"] > wf["base_rate_outperform"]).fillna(False).mean()) if len(wf) > 0 else float("nan")

    return {
        "benchmark_window": float(benchmark_window),
        "n_features": float(len(feature_set)),
        "wf_outperform_accuracy_acted_mean": acted_mean,
        "wf_base_rate_outperform_mean": base_mean,
        "wf_acted_vs_base_consistency": consistency,
        "wf_coverage_mean": float(wf["coverage"].mean()) if len(wf) > 0 else float("nan"),
    }


# Ordered checks requested.
feature_set_current = ["ret_1", "ret_5", "rel_to_ma_5", "rel_to_ma_20", "vol_20"]
feature_set_compact = ["ret_1", "rel_to_ma_20", "vol_20"]

check_rows = []

# 1) Acted-only signal check on current benchmark.
check_rows.append(run_config_check(raw_df, benchmark_window=60, feature_set=feature_set_current))

# 2) Benchmark window sensitivity check.
check_rows.append(run_config_check(raw_df, benchmark_window=20, feature_set=feature_set_current))
check_rows.append(run_config_check(raw_df, benchmark_window=30, feature_set=feature_set_current))

# 3) Smaller feature-set check (using better benchmark windows).
check_rows.append(run_config_check(raw_df, benchmark_window=20, feature_set=feature_set_compact))
check_rows.append(run_config_check(raw_df, benchmark_window=30, feature_set=feature_set_compact))

check_df = pd.DataFrame(check_rows)
check_df = check_df.sort_values(["benchmark_window", "n_features"]).reset_index(drop=True)

checks_path = Path(CONFIG.output_dir) / "auto_data_analyst_polished_ordered_checks.csv"
check_df.to_csv(checks_path, index=False)

print(check_df)
print(f"Saved ordered checks: {checks_path}")